# 04 - Figure 4: killed transport

**Question.** How do run age, recent collective order, and a smooth
late-time termination contribution combine to determine run survival?

| Panel | Analysis |
|---|---|
| A | Empirical interval termination probability and fitted hazard components |
| B | Termination probability stratified by recent order state |
| C | Empirical and model-reconstructed duration survivor functions |

Cache mode recomputes panel tables from interval and cross-fitted model
caches. Set `REFIT_FIGURE4_FIGURE5_MODELS = True` only when a full local
model refit is intended.

In [ ]:
from pathlib import Path
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


# Locate the repository before importing its analysis package. This works when
# Jupyter starts from either the repo root or this notebook directory.
_start = Path.cwd()
ROOT = next(
    path for path in (_start, *_start.parents)
    if (path / "analysis" / "levy_paper").is_dir()
    and (path / "requirements.txt").is_file()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from analysis.levy_paper.util.publication_notebook_utils import (
    PRIMARY_CACHE_SUFFIX,
    cache_path as make_cache_path,
    csv_shapes,
    display_live_or_frozen,
    file_status,
    hazard_support_summary,
    load_processed_cache,
    order_state_summary,
    panel_inventory,
    publication_paths,
    relative_path,
    resolve_data_mode,
    table_inventory,
    transition_row_sum_audit,
    transport_run_summary,
)

PATHS = publication_paths(ROOT)
LEVY_DIR = PATHS["levy_dir"]
DATA_DIR = PATHS["data_dir"]
PRIMARY_CACHE_DIR = PATHS["primary_cache_dir"]
FINAL_FIGURES = PATHS["final_figures"]
SOURCE_DATA = PATHS["source_data"]
SUPPLEMENT = PATHS["supplement"]
CACHE_SUFFIX = PRIMARY_CACHE_SUFFIX

# DATA_MODE options:
#   "auto"     use processed caches when all required files exist;
#              otherwise use tracked reviewer tables/frozen figures
#   "cache"    require processed caches and fail clearly if they are missing
#   "reviewer" use only tracked public artefacts
DATA_MODE = "auto"
BUILD_FIGURE = True
SAVE_FIGURE_OUTPUTS = True
DISPLAY_FROZEN_OUTPUT = True
REBUILD_CACHE_FROM_AWS = False
REFIT_FIGURE4_FIGURE5_MODELS = False
USE_VERSIONED_FINAL_FIGURE4_FIT = True


def rel(path):
    return relative_path(path, ROOT)


def show_file_status(paths):
    return file_status(paths, ROOT)


def show_csv_shapes(paths):
    return csv_shapes(paths, ROOT)


def cache_path(stem):
    return make_cache_path(PRIMARY_CACHE_DIR, stem, CACHE_SUFFIX)


def show_figure(fig, frozen_path, width=1100):
    return display_live_or_frozen(
        fig,
        frozen_path,
        display_frozen=DISPLAY_FROZEN_OUTPUT,
        width=width,
    )

## 1. Select the model-cache pathway

In [ ]:
model_cache = DATA_DIR / "processed" / "figure4_fig5_crossfitted" / "out_of_fold_interval_predictions.parquet"
if REFIT_FIGURE4_FIGURE5_MODELS:
    if DATA_MODE == "reviewer":
        raise ValueError("Model refitting is incompatible with reviewer mode.")
    if not cache_path("hazard_intervals").exists():
        raise FileNotFoundError("The hazard-interval cache is required for refitting.")
    subprocess.run([sys.executable, str(LEVY_DIR / "scripts" / "build_figure4_figure5_model_cache.py")], cwd=ROOT, check=True)
else:
    print("model refit skipped", rel(model_cache), model_cache.exists())

resolved_mode = resolve_data_mode(DATA_MODE, [model_cache])
print("resolved_data_mode", resolved_mode)
display(show_file_status([cache_path("hazard_intervals"), model_cache]))

## 2. Derive the three panel tables

In [ ]:
from analysis.levy_paper.scripts import create_final_fig4_fig5_polished as figure45

if resolved_mode == "cache":
    if REFIT_FIGURE4_FIGURE5_MODELS or not USE_VERSIONED_FINAL_FIGURE4_FIT:
        rebuilt_panel_a_audit = model_cache.parent / "figure4_panelA_hazard_audit.csv"
        if not rebuilt_panel_a_audit.exists():
            raise FileNotFoundError(f"Missing rebuilt Figure 4 hazard audit: {rel(rebuilt_panel_a_audit)}")
        figure45.PANEL_A_AUDIT = rebuilt_panel_a_audit
    panel_bundle = figure45.build_main_panel_sources()
    panel_a = panel_bundle["figure4a"]
    panel_b = panel_bundle["figure4b"]
    panel_c = panel_bundle["figure4c"]
    hazard_parameters = panel_bundle["tables"]["hazard_parameters"]
    validation_metrics = panel_bundle["tables"]["validation_metrics"]
    fit_mode = "versioned_final_smooth_late_fit" if USE_VERSIONED_FINAL_FIGURE4_FIT and not REFIT_FIGURE4_FIGURE5_MODELS else "rebuilt_model_cache_fit"
    panel_source_mode = f"derived_from_interval_and_crossfit_caches_with_{fit_mode}"
else:
    panel_a = pd.read_csv(SOURCE_DATA / "figure4A_source_data.csv")
    panel_b = pd.read_csv(SOURCE_DATA / "figure4B_source_data.csv")
    panel_c = pd.read_csv(SOURCE_DATA / "figure4C_source_data.csv")
    hazard_parameters = pd.read_csv(SOURCE_DATA / "crossfit_hazard_parameters.csv")
    validation_metrics = pd.read_csv(SOURCE_DATA / "final_crossfitted_validation_metrics.csv")
    panel_source_mode = "tracked_publication_source_table_fallback"

print("panel_source_mode", panel_source_mode)
display(panel_inventory({"Panel A": panel_a, "Panel B": panel_b, "Panel C": panel_c}))

## 3. Panel A - empirical risk sets and fitted support

In [ ]:
display(hazard_support_summary(panel_a))
panel_a_columns = [
    column for column in [
        "age_s", "n_at_risk", "n_terminated", "empirical_estimate",
        "ci_low", "ci_high", "age_only_baseline", "full_hazard_model",
        "sparse_flag", "fitted_flag",
    ] if column in panel_a
]
display(panel_a[panel_a_columns].head(15))

## 4. Panel B - recent-order termination strata

In [ ]:
panel_b_summary = panel_b.groupby("state", observed=True).agg(
    age_bins=("age_s", "size"),
    interval_exposure=("n_at_risk", "sum"),
    terminations=("n_terminated", "sum"),
    min_age_s=("age_s", "min"),
    max_age_s=("age_s", "max"),
).reset_index()
display(panel_b_summary)
display(panel_b[[column for column in ["age_s", "state", "n_at_risk", "n_terminated", "empirical_estimate", "ci_low", "ci_high"] if column in panel_b]].head(15))

## 5. Panel C - interval-consistent survival reconstruction

In [ ]:
panel_c_summary = panel_c.groupby("series", observed=True).agg(
    points=("age_s", "size"),
    min_age_s=("age_s", "min"),
    max_age_s=("age_s", "max"),
    min_survival=("estimate", "min"),
).reset_index()
display(panel_c_summary)

## 6. Fitted parameters and held-out validation

In [ ]:
display(hazard_parameters.head(20))
display(validation_metrics)

## 7. Construct the publication figure

In [ ]:
fig = None
if BUILD_FIGURE:
    figure45.configure_style()
    created = []
    fig = figure45.plot_figure4(
        panel_a, panel_b, panel_c, created,
        close_figure=False,
        save_outputs=SAVE_FIGURE_OUTPUTS,
    )
display_mode = show_figure(fig, FINAL_FIGURES / "figure4_killed_transport.png")
print("figure_display_mode", display_mode)